<figure>
  <IMG SRC="https://upload.wikimedia.org/wikipedia/commons/thumb/d/d5/Fachhochschule_Südwestfalen_20xx_logo.svg/320px-Fachhochschule_Südwestfalen_20xx_logo.svg.png" WIDTH=250 ALIGN="right">
</figure>

# Machine Learning
### Sommersemester 2026
Prof. Dr. Stefan Goetze

## Fashion MNIST mit Keras
**Nach dem Tutorial von Google:** https://www.tensorflow.org/tutorials/keras/classification

In diesem Notebook geht es um das selbe Thema wie schon beim Aufgabenblatt zur Logistischen Regression, nämlich den *Fashion MNIST* Datensatz.
An dieser Stelle wollen wir allerdings statt eine Multi-Klassen Logistischen Regression ein Neuronales Netz einsetzen.

Um den Code so kompakt wie möglich zu halten, verwenden wir die Keras API.
Zusätzlich benötigen Wir Funktionen aus NumPy und Matplotlib.

In [ ]:
# TensorFlow and tf.keras
import tensorflow as tf
from tensorflow import keras

# Helper libraries
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)
%load_ext tensorboard

Im Arbeitsblatt zur Logistischen Regression habe wir den Datensatz noch über eine URL aus einem Cloud Speicher heruntergeladen.
Der Fashion MNIST Datensatz ist aber ebenfalls als Standard-Beispiel über die Keras Bibliothek erhältlich.
Daher können wir ihn komfortabel über einen Keras-Aufruf [`keras.datasets.fashion_mnist.load_data()`](https://www.tensorflow.org/api_docs/python/tf/keras/datasets/fashion_mnist/load_data) herunterladen und direkt auf Trainings- und Testdatensätze aufteilen:

In [ ]:
fashion_mnist = keras.datasets.fashion_mnist
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

train_images = train_images.reshape(train_images.shape[0], train_images.shape[1], train_images.shape[2], 1)
test_images = test_images.reshape(test_images.shape[0], test_images.shape[1], test_images.shape[2], 1)

Auf die Labels wenden wir ein [`one_hot`](https://www.tensorflow.org/api_docs/python/tf/one_hot)-Encoding an.

In [ ]:
train_labels_ohe = tf.one_hot(train_labels, depth=10)
test_labels_ohe = tf.one_hot(test_labels, depth=10)

iLabel = np.random.randint(0,test_labels.shape[0])
print('Example for label no ' + str(iLabel) +' from train set: ' + str(train_labels[iLabel]) + ' (which is after one-hot-encoding:)' )
print(train_labels_ohe[iLabel])

print('Example label no ' + str(iLabel) +' from test set: ' + str(test_labels[iLabel]) + ' (which is after one-hot-encoding:)' )
print(test_labels_ohe[iLabel])

Der Trainingsdatensatz besteht aus $60000$ Bildern und der Testdatensatz aus $10000$ Bildern, mit jeweils einer Dimension von $28 \times 28$ Pixeln.

In [ ]:
print(train_images.shape)
print(test_images.shape)

Natürlich können wir uns ein zufälliges ausgewähltes Bild ansehen:

In [ ]:
plt.imshow(train_images[iLabel].reshape(28,28))
plt.colorbar();

Die 28x28 Pixel großen Bilder bestehen aus 8-bit Grauwerten.
Um die Piwelwerte in den Bereich $[0,1]$ zu skalieren, teilen wir alle Pixel durch 255.

In [ ]:
print("Minimum value of train set (before normalisation): " + str(np.min(train_images)))
print("Maximum value of test set (before normalisation): " + str(np.max(test_images)))

#Pixelwerte nach [0,1] skalieren
train_images = train_images / 255.0
test_images = test_images / 255.0

print("Minimum value of train set (after normalisation): " + str(np.min(train_images)))
print("Maximum value of test set (after normalisation): " + str(np.max(test_images)))

In [ ]:
plt.imshow(train_images[iLabel].reshape(28,28), cmap='gray')
plt.colorbar();

Nun erzeugen wir ein [sequentielles Keras Modell](https://keras.io/guides/sequential_model/), dazu verwenden wir die Funktion [`keras.Sequential()`](https://www.tensorflow.org/api_docs/python/tf/keras/Sequential).

Zu diesem Modell können wir nun mit [`model.add()`](https://keras.io/guides/sequential_model/) Schichten hinzufügen.
Entwerfen Sie selbst eine Mehrschichtiges neuronales Netz.
Wählen Sie die Anzahl der Neuronen und die Aktivierungsfunktionen der einzelnen Schichten aus.

In [ ]:
#Modell definieren
model = tf.keras.Sequential()
# Must define the input shape in the first layer of the neural network
model.add(tf.keras.layers.Input(shape=(28, 28, 1)))
model.add(tf.keras.layers.Conv2D(filters=64, kernel_size=2, padding='same', activation='relu')) 
model.add(tf.keras.layers.MaxPooling2D(pool_size=2))
model.add(tf.keras.layers.Dropout(0.3))
model.add(tf.keras.layers.Conv2D(filters=32, kernel_size=2, padding='same', activation='relu'))
model.add(tf.keras.layers.MaxPooling2D(pool_size=2))
model.add(tf.keras.layers.Dropout(0.3))
model.add(tf.keras.layers.Flatten())
model.add(tf.keras.layers.Dense(256, activation='relu'))
model.add(tf.keras.layers.Dropout(0.5))
model.add(tf.keras.layers.Dense(10, activation='softmax'))
# Take a look at the model summary
model.summary()

Keras bietet die Möglichkeit, Modelle zu [speichern und später wieder zu laden](https://keras.io/api/models/model_saving_apis/model_saving_and_loading/). Der folgende Code-Block kann verwendet werden um ggf. zu einem früheren Zeitpunkt gespeicherte Modell-Dateien zu löschen. Das Modell kann im [`.h5`-Format](https://de.wikipedia.org/wiki/Hierarchical_Data_Format) oder im nativen [`.keras`-Datenformat](https://www.tensorflow.org/guide/keras/serialization_and_saving) vorliegen.

In [ ]:
# delete saved model files and logs in case you want to start fresh
bDeleteModelFilesAndLogs = True
if bDeleteModelFilesAndLogs==True:
    !rm ./FashionMNIST_CNN.h5
    !rm ./FashionMNIST_CNN.keras
    !rm -rf ./logs/*
    print("Model files and logs deleted")

In [ ]:
import os
reuse = True
model_file_extension = ".keras" # ".h5"
if(reuse == True and os.path.exists("FashionMNIST_CNN"+model_file_extension)):
    print('Loading exising FashionMNIST_CNN'+model_file_extension+' model')
    model = keras.models.load_model("FashionMNIST_CNN"+model_file_extension)
else:
    #Modell erzeugen
    model.compile(optimizer='Adam',
                  loss='sparse_categorical_crossentropy',
                  #loss='categorical_crossentropy',
                  metrics=['accuracy'])

# create logging directory for TensorBoard
import datetime
logdir = os.path.join("logs", datetime.datetime.now().strftime("%Y%m%d-%H%M%S")) # create folder name
tensorboard_callback = tf.keras.callbacks.TensorBoard(logdir, histogram_freq=1)

In [ ]:
#Modell trainieren
model.fit(train_images, train_labels,
          epochs=5,
          #validation_data=(test_images, test_labels),
          callbacks=[tensorboard_callback]
          )

Wir können unser Modell im [`.h5`-Format](https://de.wikipedia.org/wiki/Hierarchical_Data_Format) oder im nativen [`.keras`-Datenformat](https://www.tensorflow.org/guide/keras/serialization_and_saving) speichern.

In [ ]:
model.save("FashionMNIST_CNN"+model_file_extension)

In [ ]:
#Trainiertes Modell auswerten
test_loss, test_acc = model.evaluate (test_images, test_labels)
print('Test accuracy:', test_acc)

Die `log`-Dateien können mit [TensorBoard](https://keras.io/api/callbacks/tensorboard/) visualisiert werden.

In [ ]:
%tensorboard --logdir logs

Definieren Sie Ihr Modell erneut mit der [Funktionalen API von Keras](https://keras.io/guides/functional_api/).

In [ ]:
#Funktionale abhängigkeiten
inputs = keras.Input(shape=(28, 28))

finputs = keras.layers.Flatten()(inputs)
l1 = keras.layers.Dense(128, activation=tf.nn.relu)(finputs)
outputs = keras.layers.Dense(10, activation=tf.nn.softmax)(l1)

#Modell definieren
model = keras.Model(inputs, outputs)

#Modell erzeugen
model.compile(optimizer='sgd',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

In [ ]:
# Take a look at the model summary
model.summary()

In [ ]:
# printing some information about the data
print("Type of train_images: " + str(train_images.dtype))
print("Shape of train_images: " + str(train_images.shape))

print("Type of test_images: " + str(test_images.dtype))
print("Shape of test_images: " + str(test_images.shape))

train_labels

In [ ]:
#Modell trainieren
model.fit(train_images, train_labels, epochs=5)

#Trainiertes Modell auswerten
test_loss, test_acc = model.evaluate (test_images, test_labels)
print('Test accuracy:', test_acc)

Beiträge zu diesem Notebook: [Heiner Giefers](https://github.com/hgiefers), [Stefan Goetze](https://github.com/Stefan-Goe)